# Evaluación del Modelo BI-RADS de Hugging Face

Este notebook te permitirá evaluar completamente el modelo de clasificación de mamografías BI-RADS, calculando métricas como accuracy, recall, precision, F1-score y más.

## Características del Modelo:
- **Modelo**: Enterwar99/MODEL_MAMMOGRAFII
- **Arquitectura**: ResNet-18 modificado
- **Clases**: BI-RADS 1, 2, 3, 4, 5
- **Entrada**: Imágenes de mamografías 224x224 RGB

## 1. Instalación y Importación de Librerías

Primero instalamos e importamos todas las librerías necesarias para la evaluación del modelo.

In [ ]:
# Instalación de librerías (ejecutar solo si es necesario)
# !pip install torch torchvision huggingface-hub scikit-learn matplotlib seaborn pandas numpy

# Importaciones principales
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import torch.nn as nn
from huggingface_hub import hf_hub_download

# Importaciones para métricas y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
import os
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurar matplotlib para mejor visualización
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("✅ Todas las librerías importadas correctamente")
print(f"🖥️  Dispositivo disponible: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. Configuración y Carga del Modelo de Hugging Face

Cargamos el modelo de clasificación BI-RADS desde Hugging Face Hub.

In [ ]:
# Configuración del modelo
REPO_ID = "Enterwar99/MODEL_MAMMOGRAFII"
FILENAME = "best_model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Nombres de las clases
class_names = ['BI-RADS 1', 'BI-RADS 2', 'BI-RADS 3', 'BI-RADS 4', 'BI-RADS 5']

def get_model_architecture():
    """Crear la arquitectura del modelo ResNet-18 modificado"""
    model = models.resnet18(weights=None)
    num_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_feats, 5)  # 5 clases BI-RADS
    )
    return model

# Transformaciones de imagen (mismas que se usaron durante el entrenamiento)
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

print("🔄 Descargando y cargando modelo desde Hugging Face...")
try:
    # Descargar modelo desde Hugging Face
    model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
    
    # Crear y cargar modelo
    model = get_model_architecture()
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    print(f"✅ Modelo cargado exitosamente en {device}")
    print(f"📁 Modelo descargado en: {model_path}")
    print(f"🧠 Arquitectura: ResNet-18 con {sum(p.numel() for p in model.parameters())} parámetros")
    
except Exception as e:
    print(f"❌ Error cargando modelo: {e}")
    raise

## 3. Preparación del Dataset de Prueba

Aquí puedes cargar tus imágenes de prueba y sus etiquetas correspondientes.

In [ ]:
# Configurar la ruta a tus datos de prueba
# Modifica esta ruta según tu configuración
TEST_DATA_PATH = input("📁 Ingresa la ruta al directorio con imágenes de prueba: ").strip()

# Función para procesar una imagen
def process_image(image_path):
    """Procesar una imagen para predicción"""
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
        return image_tensor
    except Exception as e:
        print(f"❌ Error procesando {image_path}: {e}")
        return None

# Función para obtener predicción
def predict_image(image_tensor):
    """Obtener predicción del modelo"""
    with torch.no_grad():
        outputs = model(image_tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probs, 1)
        
        return {
            'predicted_class': predicted_idx.item(),
            'confidence': float(confidence),
            'probabilities': probs.squeeze().cpu().numpy()
        }

# Cargar imágenes de prueba
def load_test_data(data_path):
    """Cargar datos de prueba desde directorio"""
    image_extensions = ('.png', '.jpg', '.jpeg', '.dcm', '.tif', '.tiff')
    image_files = [f for f in os.listdir(data_path) 
                   if f.lower().endswith(image_extensions)]
    
    if not image_files:
        print("❌ No se encontraron imágenes en el directorio")
        return None, None
    
    print(f"📸 Encontradas {len(image_files)} imágenes")
    
    # Verificar si existe archivo de ground truth
    gt_file = os.path.join(data_path, 'ground_truth.json')
    ground_truth = {}
    
    if os.path.exists(gt_file):
        with open(gt_file, 'r') as f:
            ground_truth = json.load(f)
        print(f"✅ Cargado ground truth desde {gt_file}")
    else:
        print("📝 No se encontró ground truth. Creando archivo interactivo...")
        # Crear ground truth interactivamente
        for image_file in image_files:
            while True:
                try:
                    birads = int(input(f"Etiqueta BI-RADS para {image_file} (1-5): "))
                    if 1 <= birads <= 5:
                        ground_truth[image_file] = birads - 1  # Convertir a índice 0-4
                        break
                    else:
                        print("❌ Ingresa un número entre 1 y 5")
                except ValueError:
                    print("❌ Ingresa un número válido")
        
        # Guardar ground truth
        with open(gt_file, 'w') as f:
            json.dump(ground_truth, f, indent=2)
        print(f"💾 Ground truth guardado en {gt_file}")
    
    return image_files, ground_truth

# Ejecutar carga de datos solo si la ruta existe
if os.path.exists(TEST_DATA_PATH):
    image_files, ground_truth = load_test_data(TEST_DATA_PATH)
    if image_files:
        print(f"✅ Dataset de prueba cargado: {len(image_files)} imágenes")
    else:
        print("❌ Error cargando dataset")
else:
    print(f"❌ La ruta {TEST_DATA_PATH} no existe")
    print("💡 Puedes usar las imágenes de ejemplo en backend/images/ para probar")

## 4. Generación de Predicciones

Procesamos todas las imágenes y generamos predicciones del modelo.

In [ ]:
# Generar predicciones para todas las imágenes
def generate_all_predictions():
    """Generar predicciones para todo el dataset"""
    predictions = []
    true_labels = []
    probabilities = []
    image_names = []
    
    print("🔄 Generando predicciones...")
    
    for i, (image_name, true_label) in enumerate(ground_truth.items()):
        image_path = os.path.join(TEST_DATA_PATH, image_name)
        
        if os.path.exists(image_path):
            # Procesar imagen
            image_tensor = process_image(image_path)
            if image_tensor is not None:
                # Obtener predicción
                result = predict_image(image_tensor)
                
                predictions.append(result['predicted_class'])
                true_labels.append(true_label)
                probabilities.append(result['probabilities'])
                image_names.append(image_name)
                
                # Mostrar progreso
                predicted_birads = result['predicted_class'] + 1
                true_birads = true_label + 1
                correct = "✅" if result['predicted_class'] == true_label else "❌"
                
                print(f"{i+1:2d}. {image_name}: Pred={predicted_birads}, Real={true_birads}, Conf={result['confidence']:.3f} {correct}")
    
    print(f"\n✅ Predicciones completadas: {len(predictions)} imágenes procesadas")
    return np.array(predictions), np.array(true_labels), np.array(probabilities), image_names

# Generar predicciones solo si tenemos datos
if 'image_files' in locals() and image_files:
    predictions, true_labels, probabilities, processed_images = generate_all_predictions()
    
    # Crear DataFrame con resultados
    results_df = pd.DataFrame({
        'Imagen': processed_images,
        'Etiqueta_Real': [f'BI-RADS {label+1}' for label in true_labels],
        'Prediccion': [f'BI-RADS {pred+1}' for pred in predictions],
        'Correcto': predictions == true_labels,
        'Confianza': [prob[pred] for prob, pred in zip(probabilities, predictions)]
    })
    
    print(f"\n📊 Resumen inicial:")
    print(f"Total de imágenes: {len(predictions)}")
    print(f"Predicciones correctas: {sum(predictions == true_labels)}")
    print(f"Accuracy inicial: {sum(predictions == true_labels) / len(predictions):.3f}")
    
    # Mostrar primeras predicciones
    print(f"\n📋 Primeras 5 predicciones:")
    display(results_df.head())
else:
    print("⚠️ No hay datos para procesar. Ejecuta primero la celda de carga de datos.")

## 5. Cálculo de Métricas de Accuracy

Calculamos la precisión general del modelo.

In [ ]:
# Calcular métricas de accuracy
if 'predictions' in locals():
    # Accuracy general
    accuracy = accuracy_score(true_labels, predictions)
    
    # Accuracy por clase
    accuracy_per_class = []
    for i in range(5):
        class_mask = true_labels == i
        if np.sum(class_mask) > 0:
            class_accuracy = accuracy_score(true_labels[class_mask], predictions[class_mask])
            accuracy_per_class.append(class_accuracy)
        else:
            accuracy_per_class.append(0.0)
    
    # Crear visualización de accuracy
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Gráfico 1: Accuracy general
    ax1.bar(['Accuracy General'], [accuracy], color='skyblue', alpha=0.8)
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Score')
    ax1.set_title(f'Accuracy General del Modelo\n{accuracy:.3f} ({accuracy*100:.1f}%)')
    
    # Agregar texto con el valor
    ax1.text(0, accuracy + 0.02, f'{accuracy:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Gráfico 2: Accuracy por clase
    classes = [f'BI-RADS {i+1}' for i in range(5)]
    colors = plt.cm.viridis(np.linspace(0, 1, 5))
    bars = ax2.bar(classes, accuracy_per_class, color=colors, alpha=0.8)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Accuracy por Clase BI-RADS')
    ax2.tick_params(axis='x', rotation=45)
    
    # Agregar valores en las barras
    for bar, acc in zip(bars, accuracy_per_class):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Mostrar resultados detallados
    print("🎯 MÉTRICAS DE ACCURACY")
    print("=" * 50)
    print(f"📊 Accuracy General: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"✅ Predicciones correctas: {sum(predictions == true_labels)}/{len(predictions)}")
    print(f"❌ Predicciones incorrectas: {sum(predictions != true_labels)}/{len(predictions)}")
    print("\n📈 Accuracy por Clase:")
    
    for i, (class_name, acc) in enumerate(zip(class_names, accuracy_per_class)):
        count = sum(true_labels == i)
        print(f"   {class_name}: {acc:.4f} ({acc*100:.1f}%) - {count} muestras")
    
    # Estadísticas adicionales
    print(f"\n📋 Estadísticas Adicionales:")
    print(f"   Mejor clase: {class_names[np.argmax(accuracy_per_class)]} ({max(accuracy_per_class):.3f})")
    print(f"   Peor clase: {class_names[np.argmin(accuracy_per_class)]} ({min(accuracy_per_class):.3f})")
    print(f"   Desviación estándar de accuracy: {np.std(accuracy_per_class):.3f}")
    
else:
    print("⚠️ No hay predicciones disponibles. Ejecuta primero la celda de predicciones.")

## 6. Cálculo de Métricas de Recall

El recall mide qué tan bien el modelo identifica cada clase (sensibilidad).

In [ ]:
# Calcular métricas de recall
if 'predictions' in locals():
    # Recall macro (promedio de todas las clases)
    recall_macro = recall_score(true_labels, predictions, average='macro', zero_division=0)
    
    # Recall micro (considera todas las muestras)
    recall_micro = recall_score(true_labels, predictions, average='micro', zero_division=0)
    
    # Recall por clase
    recall_per_class = recall_score(true_labels, predictions, average=None, zero_division=0)
    
    # Visualización de recall
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Gráfico 1: Recall macro vs micro
    recall_types = ['Macro', 'Micro']
    recall_values = [recall_macro, recall_micro]
    colors = ['lightcoral', 'lightblue']
    
    bars1 = ax1.bar(recall_types, recall_values, color=colors, alpha=0.8)
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Recall')
    ax1.set_title('Recall Macro vs Micro')
    
    # Agregar valores en las barras
    for bar, val in zip(bars1, recall_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Gráfico 2: Recall por clase
    classes = [f'BI-RADS {i+1}' for i in range(5)]
    colors = plt.cm.plasma(np.linspace(0, 1, 5))
    bars2 = ax2.bar(classes, recall_per_class, color=colors, alpha=0.8)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Recall')
    ax2.set_title('Recall por Clase BI-RADS')
    ax2.tick_params(axis='x', rotation=45)
    
    # Agregar valores en las barras
    for bar, recall in zip(bars2, recall_per_class):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{recall:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Mostrar resultados detallados
    print("🔍 MÉTRICAS DE RECALL (SENSIBILIDAD)")
    print("=" * 50)
    print(f"📊 Recall Macro: {recall_macro:.4f} ({recall_macro*100:.2f}%)")
    print(f"📊 Recall Micro: {recall_micro:.4f} ({recall_micro*100:.2f}%)")
    print("\n📈 Recall por Clase:")
    
    for i, (class_name, recall) in enumerate(zip(class_names, recall_per_class)):
        # Calcular verdaderos positivos y falsos negativos
        tp = sum((true_labels == i) & (predictions == i))
        fn = sum((true_labels == i) & (predictions != i))
        total_real = sum(true_labels == i)
        
        print(f"   {class_name}:")
        print(f"     Recall: {recall:.4f} ({recall*100:.1f}%)")
        print(f"     TP: {tp}, FN: {fn}, Total real: {total_real}")
        
        if recall < 0.5:
            print(f"     ⚠️  Recall bajo - El modelo tiene dificultad detectando esta clase")
        elif recall > 0.8:
            print(f"     ✅ Recall alto - El modelo detecta bien esta clase")
    
    # Interpretación del recall
    print(f"\n💡 INTERPRETACIÓN:")
    print(f"   • Recall Macro: Promedio del recall de todas las clases")
    print(f"   • Recall Micro: Recall considerando todas las muestras como un conjunto")
    print(f"   • Un recall alto significa que el modelo detecta bien los casos positivos")
    print(f"   • Un recall bajo indica muchos falsos negativos (casos no detectados)")
    
    # Identificar clases problemáticas
    low_recall_classes = [i for i, recall in enumerate(recall_per_class) if recall < 0.6]
    if low_recall_classes:
        print(f"\n⚠️  CLASES CON RECALL BAJO (<60%):")
        for i in low_recall_classes:
            print(f"   • {class_names[i]}: {recall_per_class[i]:.3f}")
            print(f"     💡 Sugerencia: Revisar más ejemplos de entrenamiento para esta clase")
    
else:
    print("⚠️ No hay predicciones disponibles. Ejecuta primero la celda de predicciones.")

## 7. Cálculo de Precision y F1-Score

La precision mide la calidad de las predicciones positivas, y el F1-Score combina precision y recall.

In [ ]:
# Calcular métricas de precision y F1-score
if 'predictions' in locals():
    # Precision
    precision_macro = precision_score(true_labels, predictions, average='macro', zero_division=0)
    precision_micro = precision_score(true_labels, predictions, average='micro', zero_division=0)
    precision_per_class = precision_score(true_labels, predictions, average=None, zero_division=0)
    
    # F1-Score
    f1_macro = f1_score(true_labels, predictions, average='macro', zero_division=0)
    f1_micro = f1_score(true_labels, predictions, average='micro', zero_division=0)
    f1_per_class = f1_score(true_labels, predictions, average=None, zero_division=0)
    
    # Visualización comparativa
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Gráfico 1: Precision macro vs micro
    precision_types = ['Macro', 'Micro']
    precision_values = [precision_macro, precision_micro]
    colors = ['lightgreen', 'lightcoral']
    
    bars1 = ax1.bar(precision_types, precision_values, color=colors, alpha=0.8)
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Precision')
    ax1.set_title('Precision Macro vs Micro')
    
    for bar, val in zip(bars1, precision_values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Gráfico 2: F1-Score macro vs micro
    f1_types = ['Macro', 'Micro']
    f1_values = [f1_macro, f1_micro]
    colors = ['gold', 'lightblue']
    
    bars2 = ax2.bar(f1_types, f1_values, color=colors, alpha=0.8)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('F1-Score')
    ax2.set_title('F1-Score Macro vs Micro')
    
    for bar, val in zip(bars2, f1_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Gráfico 3: Precision por clase
    classes = [f'BI-RADS {i+1}' for i in range(5)]
    colors = plt.cm.viridis(np.linspace(0, 1, 5))
    bars3 = ax3.bar(classes, precision_per_class, color=colors, alpha=0.8)
    ax3.set_ylim(0, 1)
    ax3.set_ylabel('Precision')
    ax3.set_title('Precision por Clase')
    ax3.tick_params(axis='x', rotation=45)
    
    for bar, prec in zip(bars3, precision_per_class):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{prec:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Gráfico 4: F1-Score por clase
    colors = plt.cm.plasma(np.linspace(0, 1, 5))
    bars4 = ax4.bar(classes, f1_per_class, color=colors, alpha=0.8)
    ax4.set_ylim(0, 1)
    ax4.set_ylabel('F1-Score')
    ax4.set_title('F1-Score por Clase')
    ax4.tick_params(axis='x', rotation=45)
    
    for bar, f1 in zip(bars4, f1_per_class):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{f1:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Tabla comparativa de todas las métricas
    metrics_df = pd.DataFrame({
        'Clase': class_names,
        'Precision': precision_per_class,
        'Recall': recall_per_class,
        'F1-Score': f1_per_class,
        'Accuracy': accuracy_per_class
    })
    
    print("📊 RESUMEN COMPLETO DE MÉTRICAS")
    print("=" * 70)
    print(f"🎯 Métricas Generales:")
    print(f"   Accuracy:     {accuracy:.4f} ({accuracy*100:.1f}%)")
    print(f"   Precision:    {precision_macro:.4f} (Macro), {precision_micro:.4f} (Micro)")
    print(f"   Recall:       {recall_macro:.4f} (Macro), {recall_micro:.4f} (Micro)")
    print(f"   F1-Score:     {f1_macro:.4f} (Macro), {f1_micro:.4f} (Micro)")
    
    print(f"\n📈 Métricas por Clase:")
    display(metrics_df.round(4))

## 8. Visualización de Métricas de Rendimiento

Creamos visualizaciones avanzadas incluyendo matriz de confusión, curvas ROC y análisis detallado.

In [ ]:
# Visualizaciones avanzadas
if 'predictions' in locals():
    # Crear figura con subplots
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Matriz de Confusión
    ax1 = plt.subplot(2, 3, 1)
    cm = confusion_matrix(true_labels, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                ax=ax1)
    ax1.set_title('Matriz de Confusión', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Predicción')
    ax1.set_ylabel('Etiqueta Real')
    
    # 2. Matriz de Confusión Normalizada
    ax2 = plt.subplot(2, 3, 2)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax2)
    ax2.set_title('Matriz de Confusión Normalizada', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Predicción')
    ax2.set_ylabel('Etiqueta Real')
    
    # 3. Distribución de Predicciones vs Reales
    ax3 = plt.subplot(2, 3, 3)
    x_pos = np.arange(5)
    true_counts = np.bincount(true_labels, minlength=5)
    pred_counts = np.bincount(predictions, minlength=5)
    
    width = 0.35
    ax3.bar(x_pos - width/2, true_counts, width, label='Etiquetas Reales', alpha=0.8, color='lightblue')
    ax3.bar(x_pos + width/2, pred_counts, width, label='Predicciones', alpha=0.8, color='lightcoral')
    ax3.set_xlabel('Clases BI-RADS')
    ax3.set_ylabel('Cantidad')
    ax3.set_title('Distribución de Clases: Real vs Predicción')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels([f'BI-RADS {i+1}' for i in range(5)])\n    ax3.legend()\n    ax3.grid(True, alpha=0.3)\n    \n    # 4. Métricas Comparativas\n    ax4 = plt.subplot(2, 3, 4)\n    metrics_names = ['Precision', 'Recall', 'F1-Score']\n    x_pos = np.arange(len(class_names))\n    width = 0.25\n    \n    ax4.bar(x_pos - width, precision_per_class, width, label='Precision', alpha=0.8)\n    ax4.bar(x_pos, recall_per_class, width, label='Recall', alpha=0.8)\n    ax4.bar(x_pos + width, f1_per_class, width, label='F1-Score', alpha=0.8)\n    \n    ax4.set_xlabel('Clases BI-RADS')\n    ax4.set_ylabel('Score')\n    ax4.set_title('Comparación de Métricas por Clase')\n    ax4.set_xticks(x_pos)\n    ax4.set_xticklabels([f'BI-RADS {i+1}' for i in range(5)])\n    ax4.legend()\n    ax4.set_ylim(0, 1.1)\n    ax4.grid(True, alpha=0.3)\n    \n    # 5. Análisis de Errores por Confianza\n    ax5 = plt.subplot(2, 3, 5)\n    confidences = [prob[pred] for prob, pred in zip(probabilities, predictions)]\n    correct_mask = predictions == true_labels\n    \n    # Histograma de confianzas para predicciones correctas e incorrectas\n    ax5.hist(np.array(confidences)[correct_mask], bins=20, alpha=0.7, \n            label='Predicciones Correctas', color='green', density=True)\n    ax5.hist(np.array(confidences)[~correct_mask], bins=20, alpha=0.7, \n            label='Predicciones Incorrectas', color='red', density=True)\n    ax5.set_xlabel('Confianza del Modelo')\n    ax5.set_ylabel('Densidad')\n    ax5.set_title('Distribución de Confianza: Correctas vs Incorrectas')\n    ax5.legend()\n    ax5.grid(True, alpha=0.3)\n    \n    # 6. Resumen de Métricas\n    ax6 = plt.subplot(2, 3, 6)\n    ax6.axis('off')\n    \n    # Crear texto con resumen\n    summary_text = f\"\"\"\n    RESUMEN DE EVALUACIÓN\n    \n    📊 MÉTRICAS GENERALES:\n    • Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)\n    • Precision (Macro): {precision_macro:.3f}\n    • Recall (Macro): {recall_macro:.3f}\n    • F1-Score (Macro): {f1_macro:.3f}\n    \n    📈 ANÁLISIS:\n    • Total de imágenes: {len(predictions)}\n    • Predicciones correctas: {sum(predictions == true_labels)}\n    • Predicciones incorrectas: {sum(predictions != true_labels)}\n    \n    🎯 MEJOR CLASE:\n    • {class_names[np.argmax(f1_per_class)]}\n    • F1-Score: {max(f1_per_class):.3f}\n    \n    ⚠️ CLASE MÁS DIFÍCIL:\n    • {class_names[np.argmin(f1_per_class)]}\n    • F1-Score: {min(f1_per_class):.3f}\n    \n    💡 CONFIANZA PROMEDIO:\n    • Correctas: {np.mean(np.array(confidences)[correct_mask]):.3f}\n    • Incorrectas: {np.mean(np.array(confidences)[~correct_mask]):.3f if sum(~correct_mask) > 0 else 'N/A'}\n    \"\"\"\n    \n    ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes, fontsize=11,\n            verticalalignment='top', bbox=dict(boxstyle=\"round,pad=0.4\", facecolor=\"lightgray\", alpha=0.8))\n    \n    plt.tight_layout()\n    plt.show()\n    \n    # Análisis detallado de errores\n    print(\"🔍 ANÁLISIS DETALLADO DE ERRORES\")\n    print(\"=\" * 60)\n    \n    # Encontrar errores más comunes\n    error_pairs = []\n    for true_label, pred_label in zip(true_labels, predictions):\n        if true_label != pred_label:\n            error_pairs.append((true_label, pred_label))\n    \n    if error_pairs:\n        from collections import Counter\n        error_counts = Counter(error_pairs)\n        print(\"❌ Errores más frecuentes:\")\n        for (true_class, pred_class), count in error_counts.most_common(5):\n            print(f\"   {class_names[true_class]} → {class_names[pred_class]}: {count} veces\")\n        \n        # Análisis de confianza en errores\n        error_confidences = [confidences[i] for i, (t, p) in enumerate(zip(true_labels, predictions)) if t != p]\n        if error_confidences:\n            print(f\"\\n📊 Confianza en predicciones incorrectas:\")\n            print(f\"   Promedio: {np.mean(error_confidences):.3f}\")\n            print(f\"   Mediana: {np.median(error_confidences):.3f}\")\n            print(f\"   Mínima: {np.min(error_confidences):.3f}\")\n            print(f\"   Máxima: {np.max(error_confidences):.3f}\")\n    \n    else:\n        print(\"🎉 ¡Perfecto! No hay errores en las predicciones.\")\n\nelse:\n    print(\"⚠️ No hay predicciones disponibles. Ejecuta primero las celdas anteriores.\")